<a href="https://colab.research.google.com/github/nikkojarabelo08/LABORATORY_EXAM/blob/main/Jarabelo_LABORATORYEXAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telecom Churn Prediction and Model Comparison

This notebook processes the ConnectX customer churn data and evaluates three machine learning models:
1. **Model A**: Logistic Regression
2. **Model B**: Decision Tree Classifier
3. **Model C**: Random Forest Classifier

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the dataset
file_path = '/content/telecom_churn.csv'
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (3333, 21)


,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [ ]:
# Identify features and target
# Assuming the target column is named 'Churn' or similar. We will detect it programmatically if possible, otherwise assume standard naming.
target_col = [col for col in df.columns if 'churn' in col.lower()][0]
print(f"Target column identified: {target_col}")

X = df.drop(columns=[target_col])
y = df[target_col]

# Handle target encoding if categorical
if y.dtype == 'object' or y.dtype == 'bool':
    y = y.astype(int)

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f"Numerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

# Define preprocessing pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Train-test split (80-20, stratified, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Target column identified: churn
Numerical columns: ['account length', 'area code', 'number vmail messages', 'total day minutes', 'total day calls', 'total day charge', 'total eve minutes', 'total eve calls', 'total eve charge', 'total night minutes', 'total night calls', 'total night charge', 'total intl minutes', 'total intl calls', 'total intl charge', 'customer service calls']
Categorical columns: ['state', 'phone number', 'international plan', 'voice mail plan']
Training set size: 2666
Testing set size: 667


In [ ]:
# 2. Define and train models using Pipelines with analytical perspective comments

# --- MODEL DEFLECTION & COMPLEXITY TRADE-OFFS ---
# Model A (Logistic Regression): Low complexity, high interpretability. Assumes linear relationship.
# Model B (Decision Tree): Medium complexity, high interpretability. Handles non-linear splits well.
# Model C (Random Forest): High complexity, low interpretability. A robust ensemble but a black box.
models = {
    'Model A (Logistic Regression)': LogisticRegression(max_iter=1000, random_state=42),
    'Model B (Decision Tree)': DecisionTreeClassifier(random_state=42),
    'Model C (Random Forest)': RandomForestClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    # Create a full pipeline with preprocessing and model
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

    # Train
    clf.fit(X_train, y_train)

    # Predict
    y_pred = clf.predict(X_test)

    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # --- BUSINESS ERROR IMPLICATIONS FOR EACH MODEL ---
    # If name == 'Model A': High False Negatives (Recall=24.7%). ConnectX misses a lot of churners.
    # If name == 'Model B': Balanced False Positives/Negatives (Recall=66.0%). Best for proactive retention.
    # If name == 'Model C': Highly conservative. Low False Positives (Precision=92.5%) but misses many actual churners (Recall=38.1%).

    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall (Sensitivity)': rec,
        'F1-score': f1
    }

# 3. Construct the comparative summary table for deployment analysis
# Model B remains the most practically feasible model to deploy due to low computational latency and highest Recall.
summary_df = pd.DataFrame(results).T
display(summary_df)

,Accuracy,Precision,Recall (Sensitivity),F1-score
Model A (Logistic Regression),0.859070,0.533333,0.247423,0.338028
Model B (Decision Tree),0.937031,0.876712,0.659794,0.752941
Model C (Random Forest),0.905547,0.925000,0.381443,0.540146


## Analytical Report: Trade-offs, Error Implications & Practical Feasibility

### 1. Trade-offs: Complexity vs. Interpretability vs. Performance
* **Model A (Logistic Regression)** is highly interpretable because its coefficients directly represent log-odds, making it easy to explain to stakeholders. However, it assumes linear relationships and underperformed significantly with an F1-score of **33.80%**.
* **Model B (Decision Tree)** offers a sweet spot with high performance (F1-score of **75.29%**) and relatively good interpretability, as its decisions can be mapped visually. However, simple decision trees can be prone to high variance (overfitting) if not pruned.
* **Model C (Random Forest)** is a complex ensemble method. It is highly robust and achieved the best Precision (**92.50%**), but its "black-box" nature makes it very difficult to interpret directly. Its Recall was unexpectedly low (**38.14%**), indicating it struggled to capture the full landscape of churning customers without hyperparameter tuning.

### 2. Implication of Errors: False Positives (FP) vs. False Negatives (FN)
In a customer retention setting, the cost of errors is asymmetric:
* **False Positives (Type I Error)**: The model flags a loyal customer as high-risk. ConnectX might offer them unnecessary discounts or incentives. While this incurs a financial cost, it maintains goodwill.
* **False Negatives (Type II Error)**: The model fails to identify a customer who is about to churn. ConnectX does nothing, and the customer leaves for a competitor. The cost of acquiring a new customer is exponentially higher than retaining an existing one, making **False Negatives much more expensive than False Positives**.
* Therefore, we must prioritize **Recall (Sensitivity)**. **Model B** is far superior here with a Recall of **65.98%** compared to Model C's **38.14%**.

### 3. Practical Feasibility of Deploying Model B
* **Feasibility**: Highly feasible. Decision Trees require very low computational latency for real-time predictions and can easily run on standard database or API infrastructure.
* **Recommendation**: While Model B is the best of the three out-of-the-box, we should optimize it using pruning (limiting `max_depth`) or leverage a tuned ensemble (like XGBoost or a tuned Random Forest) to capture its predictive power while controlling overfitting.

In [ ]:
# ==============================================================================
# ANALYTICAL REPORT COMMENTS (FOR CODE INTEGRATION)
# ==============================================================================
#
# 1. TRADE-OFFS (COMPLEXITY vs. INTERPRETABILITY vs. PERFORMANCE):
#    - Logistic Regression (Model A): High interpretability, poor performance (F1=33.8%).
#    - Decision Tree (Model B): Medium interpretability, highest performance (F1=75.3%, Recall=66.0%).
#    - Random Forest (Model C): Low interpretability (black-box), high Precision (92.5%) but low Recall (38.1%).
#
# 2. IMPLICATION OF ERRORS (FP vs. FN):
#    - False Positive (FP): Flagging a loyal customer. Result: Unnecessary discount cost.
#    - False Negative (FN): Missing a churning customer. Result: Permanent loss of customer (High churn cost).
#    - Since FN is much more expensive, we prioritize Recall. Model B is the clear choice.
#
# 3. PRACTICAL FEASIBILITY OF DEPLOYING THE BEST MODEL (Model B):
#    - Decision Trees have low computational latency, making deployment highly feasible.
#    - Real-world action: Use Model B's probabilities to trigger automated retention campaigns.
# ==============================================================================

print("Analytical discussion and trade-off comments have been documented successfully.")

Analytical discussion and trade-off comments have been documented successfully.
